In [3]:
import pandas as pd
import numpy as np
from sklearn.svm import SVR
from sklearn.preprocessing import RobustScaler, QuantileTransformer
from sklearn.compose import TransformedTargetRegressor
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error
from hyperopt import fmin, tpe, hp, Trials, STATUS_OK
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# 설정
# ============================================================
DATA_PATH = '/content/'  # Colab: 'open/' 또는 '/content/'
N_SPLITS = 10         # CV fold 수
HYPEROPT_EVALS = 100  # 하이퍼파라미터 탐색 횟수 (시간 여유 있으면 200+)

print("=" * 65)
print("  STRESS PREDICTION V5 - SVR ULTRA (Target: MAE < 0.14)")
print("=" * 65)

# ============================================================
# 1. 데이터 로드
# ============================================================
train = pd.read_csv(f'{DATA_PATH}train.csv')
test = pd.read_csv(f'{DATA_PATH}test.csv')
submit = pd.read_csv(f'{DATA_PATH}sample_submission.csv')

print(f"\nTrain: {train.shape}, Test: {test.shape}")
print(f"Target: mean={train['stress_score'].mean():.4f}, "
      f"std={train['stress_score'].std():.4f}")

y = train['stress_score'].copy()
train_ids = train['ID'].copy()
test_ids = test['ID'].copy()

# ============================================================
# 2. 전처리 (train/test 동시)
# ============================================================
print("\n[1] 전처리")

# ID, target 분리
train = train.drop(['ID', 'stress_score'], axis=1)
test = test.drop(['ID'], axis=1)

# --- 결측치 처리 ---
# mean_working: 0으로 채움
train['mean_working'] = train['mean_working'].fillna(0)
test['mean_working'] = test['mean_working'].fillna(0)

# 나머지 결측 → 'Unknown'
train = train.fillna('Unknown')
test = test.fillna('Unknown')

# --- 수동 매핑 (순서가 있는 범주형) ---
d_gender = {"F": 0, "M": 1}
d_activity = {"light": 0, "moderate": 1, "intense": 2}
d_smoke_status = {"non-smoker": 0, "ex-smoker": 1, "current-smoker": 2}
d_edu_level = {
    'high school diploma': 1,
    'bachelors degree': 2,
    'graduate degree': 3,
    'Unknown': 0
}
d_sleep_pattern = {
    'sleep difficulty': 0,
    'normal': 1,
    'oversleeping': 2
}

for df in [train, test]:
    df['gender'] = df['gender'].map(d_gender)
    df['activity'] = df['activity'].map(d_activity)
    df['smoke_status'] = df['smoke_status'].map(d_smoke_status)
    df['edu_level'] = df['edu_level'].map(d_edu_level)
    df['sleep_pattern'] = df['sleep_pattern'].map(d_sleep_pattern)

# --- One-Hot Encoding (SVR 핵심!) ---
# medical_history, family_medical_history는 One-Hot
# smoke_status도 One-Hot 추가 (순서형 + 더미 둘 다)
for prefix, col in [('mh', 'medical_history'),
                     ('fmh', 'family_medical_history'),
                     ('smo', 'smoke_status')]:
    train_dum = pd.get_dummies(train[col], prefix=prefix, dtype='int')
    test_dum = pd.get_dummies(test[col], prefix=prefix, dtype='int')
    # train/test 컬럼 맞추기
    for c in train_dum.columns:
        if c not in test_dum.columns:
            test_dum[c] = 0
    for c in test_dum.columns:
        if c not in train_dum.columns:
            train_dum[c] = 0
    test_dum = test_dum[train_dum.columns]
    train = pd.concat([train, train_dum], axis=1)
    test = pd.concat([test, test_dum], axis=1)

train = train.drop(['medical_history', 'family_medical_history', 'smoke_status'], axis=1)
test = test.drop(['medical_history', 'family_medical_history', 'smoke_status'], axis=1)

# ============================================================
# 3. 피처 엔지니어링 (검증된 것만!)
# ============================================================
print("[2] 피처 엔지니어링")

for df in [train, test]:
    # BMI (우승자도 사용, 검증됨)
    df['bmi'] = (df['weight'] / ((df['height'] / 100.0) ** 2)).round(2)

    # 혈압 파생 (의학적으로 의미 있는 것만)
    df['pulse_pressure'] = df['systolic_blood_pressure'] - df['diastolic_blood_pressure']
    df['map_bp'] = df['diastolic_blood_pressure'] + df['pulse_pressure'] / 3

    # 나이*활동량 교호작용
    df['age_activity'] = df['age'] * df['activity']

    # 나이*BMI
    df['age_bmi'] = df['age'] * df['bmi']

print(f"  Train: {train.shape}, Test: {test.shape}")
print(f"  Features: {list(train.columns)}")

X = train.values.astype(np.float64)
X_test = test.values.astype(np.float64)
y_arr = y.values.astype(np.float64)

# ============================================================
# 4. Hyperopt 하이퍼파라미터 최적화
# ============================================================
print(f"\n[3] Hyperopt 탐색 ({HYPEROPT_EVALS}회)")

def objective(params):
    """K-Fold CV로 SVR MAE 평가"""
    kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=42)
    fold_maes = []

    for tr_idx, val_idx in kf.split(X):
        X_tr, X_val = X[tr_idx], X[val_idx]
        y_tr, y_val = y_arr[tr_idx], y_arr[val_idx]

        pipe = make_pipeline(
            RobustScaler(),
            TransformedTargetRegressor(
                regressor=SVR(
                    kernel="rbf",
                    C=params["C"],
                    gamma=params["gamma"],
                    epsilon=params["epsilon"],
                    shrinking=True,
                    cache_size=500,
                    max_iter=-1
                ),
                transformer=QuantileTransformer(
                    output_distribution="normal",
                    n_quantiles=min(1000, len(y_tr))
                )
            )
        )

        pipe.fit(X_tr, y_tr)
        pred = pipe.predict(X_val)
        fold_maes.append(mean_absolute_error(y_val, pred))

    return {"loss": float(np.mean(fold_maes)), "status": STATUS_OK}


space = {
    "C":       hp.loguniform("C",     np.log(0.5),  np.log(30)),
    "gamma":   hp.loguniform("gamma", np.log(0.05), np.log(5)),
    "epsilon": hp.loguniform("epsilon", np.log(1e-5), np.log(0.1)),
}

trials = Trials()
best = fmin(
    fn=objective,
    space=space,
    algo=tpe.suggest,
    max_evals=HYPEROPT_EVALS,
    trials=trials,
    rstate=np.random.default_rng(42)
)

best_params = {
    "C": float(best["C"]),
    "gamma": float(best["gamma"]),
    "epsilon": float(best["epsilon"]),
}
best_mae = float(min(r["loss"] for r in trials.results))

print(f"\n  Best CV MAE: {best_mae:.5f}")
print(f"  Best params: C={best_params['C']:.4f}, "
      f"gamma={best_params['gamma']:.4f}, "
      f"epsilon={best_params['epsilon']:.6f}")

# ============================================================
# 5. 최적 파라미터로 최종 학습 + 예측
# ============================================================
print(f"\n[4] 최종 학습")

# --- 방법 1: Full train 학습 (기본 제출) ---
pipe_final = make_pipeline(
    RobustScaler(),
    TransformedTargetRegressor(
        regressor=SVR(
            kernel="rbf",
            C=best_params["C"],
            gamma=best_params["gamma"],
            epsilon=best_params["epsilon"],
            shrinking=True,
            cache_size=500,
            max_iter=-1
        ),
        transformer=QuantileTransformer(
            output_distribution="normal",
            n_quantiles=min(1000, len(y_arr))
        )
    )
)

pipe_final.fit(X, y_arr)
train_mae = mean_absolute_error(y_arr, pipe_final.predict(X))
print(f"  Train MAE (full): {train_mae:.8f}")

pred_single = pipe_final.predict(X_test)

# --- 방법 2: K-Fold Averaging (더 안정적) ---
print(f"\n[5] K-Fold Averaging (안정화)")
kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=42)
fold_preds = np.zeros((len(X_test), N_SPLITS))
oof_preds = np.zeros(len(y_arr))

for fold, (tr_idx, val_idx) in enumerate(kf.split(X)):
    X_tr, X_val = X[tr_idx], X[val_idx]
    y_tr, y_val = y_arr[tr_idx], y_arr[val_idx]

    pipe = make_pipeline(
        RobustScaler(),
        TransformedTargetRegressor(
            regressor=SVR(
                kernel="rbf",
                C=best_params["C"],
                gamma=best_params["gamma"],
                epsilon=best_params["epsilon"],
                shrinking=True,
                cache_size=500,
                max_iter=-1
            ),
            transformer=QuantileTransformer(
                output_distribution="normal",
                n_quantiles=min(1000, len(y_tr))
            )
        )
    )
    pipe.fit(X_tr, y_tr)
    oof_preds[val_idx] = pipe.predict(X_val)
    fold_preds[:, fold] = pipe.predict(X_test)
    fold_mae = mean_absolute_error(y_val, oof_preds[val_idx])
    print(f"  Fold {fold+1:2d} MAE: {fold_mae:.5f}")

pred_kfold = fold_preds.mean(axis=1)
oof_mae = mean_absolute_error(y_arr, oof_preds)
print(f"  OOF MAE: {oof_mae:.5f}")

# --- 방법 3: Multi-Seed Averaging (최대 안정화) ---
print(f"\n[6] Multi-Seed Averaging")
seeds = [42, 123, 456, 789, 2024, 2025, 2026, 777, 1234, 5678]
seed_preds = []

for seed in seeds:
    kf_s = KFold(n_splits=N_SPLITS, shuffle=True, random_state=seed)
    fold_p = []
    for tr_idx, val_idx in kf_s.split(X):
        pipe = make_pipeline(
            RobustScaler(),
            TransformedTargetRegressor(
                regressor=SVR(
                    kernel="rbf",
                    C=best_params["C"],
                    gamma=best_params["gamma"],
                    epsilon=best_params["epsilon"],
                    shrinking=True,
                    cache_size=500,
                    max_iter=-1
                ),
                transformer=QuantileTransformer(
                    output_distribution="normal",
                    n_quantiles=min(1000, len(y_arr[tr_idx]))
                )
            )
        )
        pipe.fit(X[tr_idx], y_arr[tr_idx])
        fold_p.append(pipe.predict(X_test))
    seed_preds.append(np.mean(fold_p, axis=0))

pred_multiseed = np.mean(seed_preds, axis=0)
print(f"  {len(seeds)} seeds × {N_SPLITS} folds = {len(seeds)*N_SPLITS}개 모델 평균 완료")

# ============================================================
# 6. 블렌딩 (Full + KFold + MultiSeed)
# ============================================================
print(f"\n[7] 최종 블렌딩")

# Full train 예측은 약간 오버핏 경향 → 비중 낮게
pred_blend = 0.2 * pred_single + 0.3 * pred_kfold + 0.5 * pred_multiseed
pred_blend = np.clip(pred_blend, 0, 1)

print(f"  pred mean={pred_blend.mean():.4f}, std={pred_blend.std():.4f}")

# ============================================================
# 7. 제출 파일 저장
# ============================================================
print(f"\n[8] 제출 파일 저장")

# V1: 블렌딩 (추천)
sub1 = submit.copy()
sub1['stress_score'] = pred_blend
sub1.to_csv(f'{DATA_PATH}submission_v5_blend.csv', index=False)
print(f"  ✅ submission_v5_blend.csv (추천)")

# V2: MultiSeed만
sub2 = submit.copy()
sub2['stress_score'] = np.clip(pred_multiseed, 0, 1)
sub2.to_csv(f'{DATA_PATH}submission_v5_multiseed.csv', index=False)
print(f"  ✅ submission_v5_multiseed.csv")

# V3: KFold만
sub3 = submit.copy()
sub3['stress_score'] = np.clip(pred_kfold, 0, 1)
sub3.to_csv(f'{DATA_PATH}submission_v5_kfold.csv', index=False)
print(f"  ✅ submission_v5_kfold.csv")

# V4: Full train (가장 공격적)
sub4 = submit.copy()
sub4['stress_score'] = np.clip(pred_single, 0, 1)
sub4.to_csv(f'{DATA_PATH}submission_v5_full.csv', index=False)
print(f"  ✅ submission_v5_full.csv")

print(f"\n{'='*65}")
print(f"  CV MAE: {best_mae:.5f} | OOF MAE: {oof_mae:.5f}")
print(f"  Best: C={best_params['C']:.4f}, gamma={best_params['gamma']:.4f}, "
      f"eps={best_params['epsilon']:.6f}")
print(f"  제출 순서: v5_blend → v5_multiseed → v5_full → v5_kfold")
print(f"{'='*65}")


# ============================================================
# [보너스] epsilon=0 버전
# ============================================================
print(f"\n[보너스] epsilon=0 고정 + C/gamma만 최적화")

def objective_eps0(params):
    kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=42)
    fold_maes = []
    for tr_idx, val_idx in kf.split(X):
        pipe = make_pipeline(
            RobustScaler(),
            TransformedTargetRegressor(
                regressor=SVR(kernel="rbf", C=params["C"],
                              gamma=params["gamma"], epsilon=0.0,
                              shrinking=True, cache_size=500),
                transformer=QuantileTransformer(
                    output_distribution="normal",
                    n_quantiles=min(1000, len(y_arr[tr_idx])))
            )
        )
        pipe.fit(X[tr_idx], y_arr[tr_idx])
        pred = pipe.predict(X[val_idx])
        fold_maes.append(mean_absolute_error(y_arr[val_idx], pred))
    return {"loss": float(np.mean(fold_maes)), "status": STATUS_OK}

space_eps0 = {
    "C":     hp.loguniform("C",     np.log(0.5),  np.log(50)),
    "gamma": hp.loguniform("gamma", np.log(0.05), np.log(5)),
}

trials2 = Trials()
best2 = fmin(fn=objective_eps0, space=space_eps0, algo=tpe.suggest,
             max_evals=HYPEROPT_EVALS, trials=trials2,
             rstate=np.random.default_rng(42))

best2_params = {"C": float(best2["C"]), "gamma": float(best2["gamma"])}
best2_mae = float(min(r["loss"] for r in trials2.results))
print(f"  eps=0 Best CV MAE: {best2_mae:.5f}")
print(f"  C={best2_params['C']:.4f}, gamma={best2_params['gamma']:.4f}")

# eps=0 최종 학습
pipe_eps0 = make_pipeline(
    RobustScaler(),
    TransformedTargetRegressor(
        regressor=SVR(kernel="rbf", C=best2_params["C"],
                      gamma=best2_params["gamma"], epsilon=0.0,
                      shrinking=True, cache_size=500),
        transformer=QuantileTransformer(
            output_distribution="normal",
            n_quantiles=min(1000, len(y_arr)))
    )
)
pipe_eps0.fit(X, y_arr)
pred_eps0 = np.clip(pipe_eps0.predict(X_test), 0, 1)

sub5 = submit.copy()
sub5['stress_score'] = pred_eps0
sub5.to_csv(f'{DATA_PATH}submission_v5_eps0.csv', index=False)
print(f"  ✅ submission_v5_eps0.csv")

# eps=0 multi-seed도
seed_preds_eps0 = []
for seed in seeds:
    kf_s = KFold(n_splits=N_SPLITS, shuffle=True, random_state=seed)
    fp = []
    for tr_idx, val_idx in kf_s.split(X):
        p = make_pipeline(
            RobustScaler(),
            TransformedTargetRegressor(
                regressor=SVR(kernel="rbf", C=best2_params["C"],
                              gamma=best2_params["gamma"], epsilon=0.0,
                              shrinking=True, cache_size=500),
                transformer=QuantileTransformer(
                    output_distribution="normal",
                    n_quantiles=min(1000, len(y_arr[tr_idx])))
            )
        )
        p.fit(X[tr_idx], y_arr[tr_idx])
        fp.append(p.predict(X_test))
    seed_preds_eps0.append(np.mean(fp, axis=0))

pred_eps0_ms = np.clip(np.mean(seed_preds_eps0, axis=0), 0, 1)
sub6 = submit.copy()
sub6['stress_score'] = pred_eps0_ms
sub6.to_csv(f'{DATA_PATH}submission_v5_eps0_multiseed.csv', index=False)
print(f"  ✅ submission_v5_eps0_multiseed.csv")

print(f"\n{'='*65}")
print(f"  총 6개 제출 파일 생성 완료!")
print(f"  eps 탐색: MAE={best_mae:.5f} | eps=0: MAE={best2_mae:.5f}")
print(f"  더 낮은 쪽의 blend/multiseed를 우선 제출하세요")
print(f"{'='*65}")

  STRESS PREDICTION V5 - SVR ULTRA (Target: MAE < 0.14)

Train: (3000, 18), Test: (3000, 17)
Target: mean=0.4821, std=0.2883

[1] 전처리
[2] 피처 엔지니어링
  Train: (3000, 29), Test: (3000, 29)
  Features: ['gender', 'age', 'height', 'weight', 'cholesterol', 'systolic_blood_pressure', 'diastolic_blood_pressure', 'glucose', 'bone_density', 'activity', 'sleep_pattern', 'edu_level', 'mean_working', 'mh_Unknown', 'mh_diabetes', 'mh_heart disease', 'mh_high blood pressure', 'fmh_Unknown', 'fmh_diabetes', 'fmh_heart disease', 'fmh_high blood pressure', 'smo_0', 'smo_1', 'smo_2', 'bmi', 'pulse_pressure', 'map_bp', 'age_activity', 'age_bmi']

[3] Hyperopt 탐색 (100회)
100%|██████████| 100/100 [16:35<00:00,  9.96s/trial, best loss: 0.13808824116937474]

  Best CV MAE: 0.13809
  Best params: C=21.2466, gamma=0.9767, epsilon=0.000010

[4] 최종 학습
  Train MAE (full): 0.00000000

[5] K-Fold Averaging (안정화)
  Fold  1 MAE: 0.13269
  Fold  2 MAE: 0.12625
  Fold  3 MAE: 0.13996
  Fold  4 MAE: 0.14583
  Fold  5 MAE: 